# ⚽ FIFA 15 → EA FC 26: tabla unificada y dashboard interactivo

**Cómo usarlo:** subí a `/content` los zips (`archive (1).zip` = FC 26, `archive (3).zip` = FIFA 15–FC 24, `archive (4).zip` = FC 25) y ejecutá las celdas en orden.

**Qué corrige esta versión**
- Las **posiciones** ahora salen de la columna real de cada dataset (antes no se cargaba ninguna y todo se deducía por atributos).
- Los atributos de **arqueros** ya no se mezclan con ritmo/tiro/pase.
- **Ligas** por coincidencia exacta (ya no se confunde Primera División argentina con LaLiga, ni Bundesliga austríaca con la alemana).
- **Liga Argentina** detectada por la liga, no por fragmentos de nombre de club.
- **Género** separado, **nacionalidad** real y **IDs de FC 25** con diagnóstico.
- **Dashboard HTML** nuevo: pestañas, filtros, equipo ideal en cancha, comparador con radar.

Cada línea de código tiene su comentario explicativo.

In [ ]:
# --- 1. IMPORTS Y CONFIGURACIÓN ----------------------------------------------
import os                                      # manejo de rutas y carpetas
import re                                      # expresiones regulares (limpiar textos, extraer IDs)
import json                                    # armar el JSON que viaja dentro del HTML
import glob                                    # buscar archivos por patrón
import zipfile                                 # descomprimir los .zip subidos a Colab
import unicodedata                             # quitar tildes al normalizar textos
import numpy as np                             # valores NaN y operaciones numéricas
import pandas as pd                            # tablas (DataFrames)

BASE = '/content'                              # carpeta de trabajo de Colab
EXTRAER = f'{BASE}/datos_extraidos'            # carpeta donde se descomprimen los zips
os.makedirs(EXTRAER, exist_ok=True)            # crea la carpeta si todavía no existe

ZIPS = {                                       # nombre lógico -> ruta del zip subido
    'fc26': f'{BASE}/archive (1).zip',         # EA FC 26 (players / outfield / goalkeepers)
    'historico': f'{BASE}/archive (3).zip',    # FIFA 15 a FC 24 (male_players.csv, ...)
    'fc25': f'{BASE}/archive (4).zip',         # EA FC 25 (male / female / all players)
}

for nombre, ruta in ZIPS.items():              # recorre cada zip
    if not os.path.exists(ruta):               # si el archivo no está subido...
        print(f'⚠️ Falta {ruta}')              # ...avisa
        continue                               # ...y sigue con el siguiente
    with zipfile.ZipFile(ruta) as z:           # abre el zip
        z.extractall(f'{EXTRAER}/{nombre}')    # extrae todo en su propia carpeta
    print(f'✅ Descomprimido: {nombre}')        # confirma

In [ ]:
# --- 2. FUNCIONES AUXILIARES --------------------------------------------------
def sin_tildes(texto):
    """Quita tildes: 'Vélez' -> 'Velez'."""
    texto = unicodedata.normalize('NFKD', str(texto))                  # separa cada letra de su tilde
    return ''.join(c for c in texto if not unicodedata.combining(c))   # descarta los caracteres de tilde

def limpiar(texto):
    """Minúsculas, sin tildes ni símbolos. Sirve para comparar nombres de ligas."""
    texto = sin_tildes(texto).lower()                  # sin tildes y en minúsculas
    texto = re.sub(r'[^a-z0-9]+', ' ', texto)          # todo lo que no es letra/número pasa a espacio
    return texto.strip()                               # saca espacios de los extremos

def buscar(carpeta, patron):
    """Busca (también en subcarpetas) el primer archivo que cumpla el patrón; None si no hay."""
    hallados = glob.glob(f'{EXTRAER}/{carpeta}/**/{patron}', recursive=True)  # lista de coincidencias
    return hallados[0] if hallados else None                                  # el primero, o None

def leer(ruta):
    """Lee un CSV completo; devuelve None si la ruta no existe."""
    if ruta is None:                                   # no se encontró el archivo
        return None                                    # devolvemos None para que el llamador decida
    return pd.read_csv(ruta, low_memory=False)         # lectura completa sin inferencia por bloques

def tomar(df, *candidatos):
    """Devuelve la primera columna que exista (sin importar mayúsculas); si ninguna existe, una serie vacía (NaN)."""
    minusculas = {c.lower(): c for c in df.columns}    # mapa 'nombre en minúscula' -> nombre real
    for cand in candidatos:                            # prueba cada nombre candidato en orden
        real = minusculas.get(cand.lower())            # busca el nombre real de la columna
        if real is not None:                           # si existe...
            return df[real]                            # ...la devuelve
    return pd.Series(np.nan, index=df.index, dtype='object')   # si no, una columna de NaN

def numero(serie, minimo=None, maximo=None):
    """Convierte a número (toma el primer número de textos como '182cm') y descarta valores fuera de rango."""
    if serie.dtype == object:                                          # si viene como texto...
        serie = serie.astype(str).str.extract(r'(\d+(?:\.\d+)?)')[0]   # ...extrae el primer número
    serie = pd.to_numeric(serie, errors='coerce')                      # a número; lo inválido pasa a NaN
    if minimo is not None:                                             # si hay mínimo...
        serie = serie.where(serie >= minimo)                           # ...descarta lo menor (queda NaN)
    if maximo is not None:                                             # si hay máximo...
        serie = serie.where(serie <= maximo)                           # ...descarta lo mayor
    return serie                                                       # serie numérica limpia

def etiqueta_edicion(version):
    """15 -> 'FIFA 15', 24 -> 'FC 24' (desde 2024 el juego pasó a llamarse EA FC)."""
    return f'FC {int(version)}' if int(version) >= 24 else f'FIFA {int(version)}'

def normalizar_genero(serie):
    """Devuelve 'F' si el texto indica mujer (female/woman) y 'M' en cualquier otro caso."""
    texto = serie.astype(str).str.strip().str.lower()              # texto en minúsculas
    es_mujer = texto.str.startswith(('f', 'w'))                    # 'female', 'woman', 'w'...
    return pd.Series(np.where(es_mujer, 'F', 'M'), index=serie.index)   # 'F' o 'M'

In [ ]:
# --- 3. CARGADORES POR FUENTE (todos devuelven las MISMAS columnas) -----------
COLUMNAS = ['player_id', 'nombre', 'edicion', 'version', 'genero', 'fecha_nacimiento', 'edad',
            'altura_cm', 'peso_kg', 'club', 'liga_original', 'nacionalidad', 'posicion_original',
            'overall', 'potencial', 'ritmo', 'tiro', 'pase', 'regate', 'defensa', 'fisico',
            'gk_estirada', 'gk_manejo', 'gk_saque', 'gk_reflejos', 'gk_posicion']   # esquema común

# ---------- EA FC 26 ----------
def cargar_fc26(carpeta):
    """Une players + outfield + goalkeepers de FC 26 (cada archivo aporta columnas distintas)."""
    players = leer(buscar(carpeta, '*fc26_players.csv'))                  # ficha general de jugadores
    if players is None:                                                   # si no está el archivo base...
        print('⚠️ No se encontró FC 26'); return None                     # ...avisa y corta
    extras = [leer(buscar(carpeta, p)) for p in ('*fc26_outfield.csv', '*fc26_goalkeepers.csv')]  # stats de campo y arqueros
    extras = [e for e in extras if e is not None]                         # descarta los que no existan
    base = players.drop_duplicates('id').set_index('id')                  # indexa por id, sin repetidos
    if extras:                                                            # si hay archivos de stats...
        stats = pd.concat(extras, ignore_index=True).drop_duplicates('id').set_index('id')  # campo + arqueros juntos
        base = stats.combine_first(base)                                  # prioriza stats y completa con la ficha general
    df = base.reset_index()                                               # 'id' vuelve a ser columna
    nombre = tomar(df, 'commonName').fillna(                              # usa el nombre común si existe...
        tomar(df, 'firstName').fillna('').astype(str) + ' ' + tomar(df, 'lastName').fillna('').astype(str))  # ...si no, nombre + apellido
    tabla = pd.DataFrame({
        'player_id': df['id'],                                            # ID propio del dataset
        'nombre': nombre.astype(str).str.strip(),                         # nombre limpio
        'edicion': 'FC 26', 'version': 26,                                # edición fija para este archivo
        'genero': normalizar_genero(tomar(df, 'gender')),                 # 'M' salvo que el dataset indique mujer
        'fecha_nacimiento': tomar(df, 'birthdate'),                       # fecha de nacimiento
        'edad': numero(tomar(df, 'age')),                                 # edad
        'altura_cm': numero(tomar(df, 'height'), 140, 220),               # altura (descarta valores absurdos)
        'peso_kg': numero(tomar(df, 'weight'), 40, 130),                  # peso (descarta valores absurdos)
        'club': tomar(df, 'team', 'club'),                                # club actual
        'liga_original': tomar(df, 'leagueName', 'league'),               # liga tal cual viene del dataset
        'nacionalidad': tomar(df, 'nationality'),                         # nacionalidad real del jugador
        'posicion_original': tomar(df, 'position'),                       # POSICIÓN REAL (antes no se cargaba)
        'overall': numero(tomar(df, 'overallRating', 'overall')),         # valoración general
        'potencial': numero(tomar(df, 'potential')),                      # potencial (si el dataset lo trae)
        'ritmo': numero(tomar(df, 'pac')),                                # ritmo
        'tiro': numero(tomar(df, 'sho')),                                 # tiro
        'pase': numero(tomar(df, 'pas')),                                 # pase
        'regate': numero(tomar(df, 'dri')),                               # regate
        'defensa': numero(tomar(df, 'def')),                              # defensa
        'fisico': numero(tomar(df, 'phy')),                               # físico
        'gk_estirada': numero(tomar(df, 'gkDiving')),                     # arqueros: estirada (columna PROPIA)
        'gk_manejo': numero(tomar(df, 'gkHandling')),                     # arqueros: manejo
        'gk_saque': numero(tomar(df, 'gkKicking')),                       # arqueros: saque
        'gk_reflejos': numero(tomar(df, 'gkReflexes')),                   # arqueros: reflejos
        'gk_posicion': numero(tomar(df, 'gkPositioning')),                # arqueros: colocación
    })
    return tabla[COLUMNAS]                                                # mismas columnas en el mismo orden

# ---------- EA FC 25 ----------
def id_desde_url(url):
    """Extrae el ID numérico del jugador desde la URL probando varios formatos."""
    if not isinstance(url, str):                                          # si no hay URL...
        return np.nan                                                     # ...no hay ID
    for patron in (r'/player/(\d+)', r'/(\d+)/?(?:\?.*)?$', r'[?&]id=(\d+)'):   # formatos posibles
        m = re.search(patron, url)                                        # intenta cada patrón
        if m:                                                             # si coincide...
            return int(m.group(1))                                        # ...devuelve el número
    return np.nan                                                         # ningún patrón funcionó

def cargar_fc25(carpeta):
    """Lee male_players.csv y female_players.csv de FC 25 (por separado, para no mezclar géneros)."""
    partes = []                                                           # acá se acumulan las tablas
    for archivo, genero in (('male_players.csv', 'M'), ('female_players.csv', 'F')):   # un archivo por género
        df = leer(buscar(carpeta, archivo))                               # intenta leerlo
        if df is None:                                                    # si no existe...
            continue                                                      # ...pasa al siguiente
        print(f'   FC 25 {archivo}: columnas con "pos" ->', [c for c in df.columns if 'pos' in c.lower()])   # ayuda a verificar la columna de posición
        ids = tomar(df, 'url').map(id_desde_url)                          # ID sacado de la URL
        clave = (tomar(df, 'Name').astype(str) + '|' + tomar(df, 'Team').astype(str) + '|'
                 + tomar(df, 'Position', 'Positions', 'Pos').astype(str) + '|' + genero)   # clave de respaldo por jugador
        respaldo = 900_000_000 + (pd.util.hash_pandas_object(clave, index=False) % 90_000_000).astype('int64')  # ID inventado pero ESTABLE
        confiable = ids.notna()                                           # True donde el ID vino de la URL
        print(f'   FC 25 {archivo}: IDs reales extraídos {confiable.mean():.0%}')   # diagnóstico
        tabla = pd.DataFrame({
            'player_id': ids.where(confiable, respaldo).astype('int64'),  # ID real, o el de respaldo si faltó
            'nombre': tomar(df, 'Name'),                                  # nombre
            'edicion': 'FC 25', 'version': 25,                            # edición fija
            'genero': genero,                                             # género según el archivo
            'fecha_nacimiento': np.nan,                                   # el dataset no la trae
            'edad': numero(tomar(df, 'Age')),                             # edad (si existe)
            'altura_cm': numero(tomar(df, 'Height'), 140, 220),           # altura en cm (descarta '6'0' mal leído)
            'peso_kg': numero(tomar(df, 'Weight'), 40, 130),              # peso en kg
            'club': tomar(df, 'Team', 'Club'),                            # club
            'liga_original': tomar(df, 'League'),                         # liga
            'nacionalidad': tomar(df, 'Nation', 'Nationality'),           # nacionalidad
            'posicion_original': tomar(df, 'Position', 'Positions', 'Pos'),   # posición real
            'overall': numero(tomar(df, 'OVR', 'Overall')),               # valoración general
            'potencial': np.nan,                                          # FC 25 no trae potencial: queda vacío (antes se copiaba el overall)
            'ritmo': numero(tomar(df, 'PAC')), 'tiro': numero(tomar(df, 'SHO')),      # ritmo y tiro
            'pase': numero(tomar(df, 'PAS')), 'regate': numero(tomar(df, 'DRI')),     # pase y regate
            'defensa': numero(tomar(df, 'DEF')), 'fisico': numero(tomar(df, 'PHY')),  # defensa y físico
            'gk_estirada': numero(tomar(df, 'GK Diving', 'DIV')),         # arqueros: estirada
            'gk_manejo': numero(tomar(df, 'GK Handling', 'HAN')),         # arqueros: manejo
            'gk_saque': numero(tomar(df, 'GK Kicking', 'KIC')),           # arqueros: saque
            'gk_reflejos': numero(tomar(df, 'GK Reflexes', 'REF')),       # arqueros: reflejos
            'gk_posicion': numero(tomar(df, 'GK Positioning', 'POS')),    # arqueros: colocación
        })
        partes.append(tabla[COLUMNAS])                                    # guarda esta tabla
    return pd.concat(partes, ignore_index=True) if partes else None       # junta hombres y mujeres

# ---------- FIFA 15 a FC 24 ----------
def cargar_historico(carpeta):
    """Lee male_players.csv / female_players.csv (FIFA 15 a FC 24) quedándose con la última actualización de cada jugador."""
    partes = []                                                           # acumulador
    deseadas = ['player_id', 'fifa_version', 'fifa_update', 'short_name', 'long_name', 'player_positions',
                'overall', 'potential', 'age', 'dob', 'height_cm', 'weight_kg', 'club_name', 'league_name',
                'nationality_name', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic',
                'gk_diving', 'gk_handling', 'gk_kicking', 'gk_reflexes', 'gk_positioning']   # columnas que nos sirven
    for archivo, genero in (('male_players.csv', 'M'), ('female_players.csv', 'F')):   # un archivo por género
        ruta = buscar(carpeta, archivo)                                   # lo busca
        if ruta is None:                                                  # si no está...
            continue                                                      # ...lo salta
        existentes = pd.read_csv(ruta, nrows=0).columns                   # lee solo el encabezado
        df = pd.read_csv(ruta, usecols=[c for c in deseadas if c in existentes], low_memory=False)   # carga solo columnas útiles que existan
        if 'fifa_update' in df.columns:                                   # si hay varias actualizaciones por versión...
            df = df.sort_values('fifa_update', ascending=False)           # ...deja primero la más reciente
        df = df.drop_duplicates(['player_id', 'fifa_version'])            # una fila por jugador y versión (la más reciente)
        version = numero(tomar(df, 'fifa_version')).astype('int64')       # número de versión (15..24)
        tabla = pd.DataFrame({
            'player_id': df['player_id'],                                 # ID oficial (sirve para seguir al jugador entre años)
            'nombre': tomar(df, 'short_name').fillna(tomar(df, 'long_name')),   # nombre corto, o el largo si falta
            'edicion': version.map(etiqueta_edicion), 'version': version, # 'FIFA 15'... 'FC 24'
            'genero': genero,                                             # género según el archivo
            'fecha_nacimiento': tomar(df, 'dob'),                         # nacimiento
            'edad': numero(tomar(df, 'age')),                             # edad
            'altura_cm': numero(tomar(df, 'height_cm'), 140, 220),        # altura
            'peso_kg': numero(tomar(df, 'weight_kg'), 40, 130),           # peso
            'club': tomar(df, 'club_name'),                               # club
            'liga_original': tomar(df, 'league_name'),                    # liga
            'nacionalidad': tomar(df, 'nationality_name'),                # nacionalidad real (antes no se usaba)
            'posicion_original': tomar(df, 'player_positions'),           # posiciones, ej. 'RW, ST, CF' (antes no se cargaba)
            'overall': numero(tomar(df, 'overall')),                      # valoración general
            'potencial': numero(tomar(df, 'potential')),                  # potencial
            'ritmo': numero(tomar(df, 'pace')), 'tiro': numero(tomar(df, 'shooting')),          # ritmo y tiro
            'pase': numero(tomar(df, 'passing')), 'regate': numero(tomar(df, 'dribbling')),     # pase y regate
            'defensa': numero(tomar(df, 'defending')), 'fisico': numero(tomar(df, 'physic')),   # defensa y físico
            'gk_estirada': numero(tomar(df, 'gk_diving')), 'gk_manejo': numero(tomar(df, 'gk_handling')),       # arqueros
            'gk_saque': numero(tomar(df, 'gk_kicking')), 'gk_reflejos': numero(tomar(df, 'gk_reflexes')),       # arqueros
            'gk_posicion': numero(tomar(df, 'gk_positioning')),           # arqueros
        })
        partes.append(tabla[COLUMNAS])                                    # guarda la tabla
    return pd.concat(partes, ignore_index=True) if partes else None       # junta hombres y mujeres

# ---------- EJECUCIÓN ----------
fuentes = {'FC 26': cargar_fc26('fc26'),                                  # carga FC 26
           'FC 25': cargar_fc25('fc25'),                                  # carga FC 25
           'FIFA 15-FC 24': cargar_historico('historico')}                # carga el histórico
for nombre, t in fuentes.items():                                         # resumen de cada fuente
    print(f'{nombre}: {0 if t is None else len(t):,} filas')              # cantidad de filas cargadas

In [ ]:
# --- 4. UNIFICAR, NORMALIZAR LIGAS Y CLASIFICAR POSICIONES --------------------
partes = [t for t in fuentes.values() if t is not None]                  # solo las fuentes que se pudieron cargar
df = pd.concat(partes, ignore_index=True)                                # una única tabla con todo
antes = len(df)                                                          # filas antes de limpiar
df = df.drop_duplicates(['player_id', 'edicion', 'genero'], keep='first')   # un registro por jugador, edición y género
print(f'Duplicados eliminados: {antes - len(df):,} | filas finales: {len(df):,}')   # resumen

# ---------- Ligas: coincidencia EXACTA (anclada con ^ y $), no por fragmentos ----------
REGLAS_LIGAS = [                                                         # (nombre canónico, patrón sobre el texto ya 'limpio')
    ('Premier League (Inglaterra)', r'^(english )?(barclays )?premier league( 1)?$'),
    ('LaLiga (España)', r'^(spain primera division( 1)?|la ?liga( ea sports| santander| bbva)?( 1)?|liga bbva)$'),
    ('Bundesliga (Alemania)', r'^(german )?bundesliga( 1)?$'),
    ('Serie A (Italia)', r'^(italian )?serie a( tim| enilive)?( 1)?$'),
    ('Ligue 1 (Francia)', r'^(french )?ligue 1( uber eats| mcdonald s| mcdonalds)?( 1)?$'),
    ('Eredivisie (Países Bajos)', r'^(holland )?eredivisie( 1)?$'),
    ('Primeira Liga (Portugal)', r'^(portuguese )?(liga zon sagres|liga nos|liga portugal( bwin| betclic)?|primeira liga)( 1)?$'),
    ('MLS (Estados Unidos)', r'^(usa )?(major league soccer|mls)( 1)?$'),
    ('Liga MX (México)', r'^(mexican )?liga (bbva )?(bancomer )?mx( 1)?$'),
    ('Liga Profesional (Argentina)', r'^(.*argentin.*|liga profesional.*|lpf|superliga.*)$'),   # cualquier nombre que contenga argentin / liga profesional / lpf / superliga
]
REGLAS_LIGAS = [(n, re.compile(p)) for n, p in REGLAS_LIGAS]            # compila cada patrón una sola vez

def normalizar_liga(original):
    """Devuelve el nombre canónico de la liga, o el original si no coincide con ninguna regla."""
    if pd.isna(original):                                                # sin dato de liga
        return 'Desconocida'                                             # etiqueta explícita
    limpio = limpiar(original)                                           # texto comparable
    for nombre, patron in REGLAS_LIGAS:                                  # prueba cada regla
        if patron.match(limpio):                                         # coincidencia exacta
            return nombre                                                # nombre canónico
    return str(original).strip()                                         # sin regla: se conserva tal cual

mapa_ligas = {l: normalizar_liga(l) for l in df['liga_original'].dropna().unique()}   # normaliza solo los valores únicos (rápido)
df['liga'] = df['liga_original'].map(mapa_ligas).fillna('Desconocida')   # aplica el mapa a toda la tabla
# ---------- Respaldo por clubes: detecta la Liga Argentina aunque el dataset la nombre distinto ----------
ARG = 'Liga Profesional (Argentina)'                                     # nombre canónico de la liga
CLUBES_LPF = {                                                           # nombres de clubes ya 'limpios' (sin tildes ni símbolos); la comparación es EXACTA
    'river plate', 'boca juniors', 'racing club', 'independiente', 'san lorenzo', 'talleres', 'talleres de cordoba',
    'estudiantes', 'estudiantes de la plata', 'velez sarsfield', 'velez', 'lanus', 'newell s old boys', 'newell s',
    'rosario central', 'huracan', 'belgrano', 'belgrano de cordoba', 'godoy cruz', 'godoy cruz antonio tomba',
    'defensa y justicia', 'defensa', 'argentinos juniors', 'argentinos jrs', 'atletico tucuman', 'banfield', 'gimnasia',
    'gimnasia y esgrima la plata', 'gimnasia la plata', 'union', 'union de santa fe', 'instituto', 'instituto de cordoba',
    'platense', 'tigre', 'barracas central', 'sarmiento', 'sarmiento de junin', 'central cordoba', 'central cordoba sde',
    'riestra', 'deportivo riestra', 'dep riestra', 'independiente rivadavia', 'ind rivadavia', 'aldosivi', 'san martin',
    'san martin de san juan', 'colon', 'colon de santa fe', 'atletico de rafaela', 'arsenal de sarandi', 'olimpo',
    'quilmes', 'temperley', 'chacarita juniors', 'patronato', 'all boys', 'lanus'}
mapa_clubes = {c: limpiar(c) for c in df['club'].dropna().unique()}      # club -> versión limpia (solo valores únicos)
df['es_club_lpf'] = df['club'].map(mapa_clubes).isin(CLUBES_LPF)         # True si el club es de la Liga Profesional
resumen = (df[df['liga'] != ARG].groupby(['edicion', 'liga_original'])['es_club_lpf']
           .agg(n='size', pct='mean').reset_index())                     # por edición y nombre de liga: filas y % de clubes argentinos
candidatas = resumen[(resumen['pct'] >= 0.5) & (resumen['n'] >= 15)]     # ligas cuyos clubes son mayoritariamente argentinos
claves = pd.MultiIndex.from_frame(df[['edicion', 'liga_original']])      # par (edición, liga) de cada fila
por_nombre = claves.isin(list(zip(candidatas['edicion'], candidatas['liga_original'])))   # filas dentro de esas ligas
sin_liga_arg = df['liga_original'].isna() & df['es_club_lpf']            # filas sin liga cuyo club es argentino
df.loc[por_nombre | sin_liga_arg, 'liga'] = ARG                          # las reasigna a la Liga Argentina
if len(candidatas):                                                      # informa qué nombres de liga se reasignaron
    print('ℹ️ Ligas reasignadas a la Liga Argentina por coincidencia de clubes:')
    print(candidatas.assign(pct=(candidatas['pct'] * 100).round(0)).to_string(index=False))
df['es_liga_arg'] = df['liga'] == ARG                                    # Liga Argentina = por liga (o por clubes si el nombre no coincidía)

# ---------- Posiciones: se usan los códigos reales del juego ----------
LINEA_POR_POSICION = {                                                   # código de posición -> línea táctica
    'GK': 'Arquero',
    'CB': 'Defensores', 'LB': 'Defensores', 'RB': 'Defensores', 'LWB': 'Defensores', 'RWB': 'Defensores', 'SW': 'Defensores',
    'CDM': 'Mediocampistas', 'CM': 'Mediocampistas', 'CAM': 'Mediocampistas', 'LM': 'Mediocampistas', 'RM': 'Mediocampistas',
    'ST': 'Delanteros', 'CF': 'Delanteros', 'LW': 'Delanteros', 'RW': 'Delanteros', 'LF': 'Delanteros', 'RF': 'Delanteros', 'SS': 'Delanteros',
}
EQUIVALENCIAS_ES = {                                                     # por si algún dataset usa códigos en español
    'POR': 'GK', 'DFC': 'CB', 'LI': 'LB', 'LD': 'RB', 'CAI': 'LWB', 'CAD': 'RWB', 'MCD': 'CDM', 'MC': 'CM',
    'MCO': 'CAM', 'MP': 'CAM', 'MI': 'LM', 'MD': 'RM', 'EI': 'LW', 'ED': 'RW', 'DC': 'ST', 'SD': 'SS',
}

def posicion_principal(texto):
    """Toma la PRIMERA posición de textos como 'RW, ST, CF' y la lleva a un código estándar."""
    if not isinstance(texto, str) or not texto.strip():                  # vacío o no es texto
        return np.nan                                                    # sin posición
    primero = re.split(r'[,;/\s]+', texto.strip().upper())[0]            # primer código del listado
    primero = EQUIVALENCIAS_ES.get(primero, primero)                     # traduce si venía en español
    return primero if primero in LINEA_POR_POSICION else np.nan          # solo códigos conocidos

mapa_pos = {p: posicion_principal(p) for p in df['posicion_original'].dropna().unique()}   # solo valores únicos
df['posicion'] = df['posicion_original'].map(mapa_pos)                   # posición estándar (ej. 'ST')
df['linea'] = df['posicion'].map(LINEA_POR_POSICION)                     # línea táctica a partir de la posición

# Respaldo SOLO para filas sin posición: si tiene stats de arquero (o ninguna stat de campo) es arquero
COLS_CAMPO = ['ritmo', 'tiro', 'pase', 'regate', 'defensa', 'fisico']                              # atributos de jugador de campo
COLS_GK = ['gk_estirada', 'gk_manejo', 'gk_saque', 'gk_reflejos', 'gk_posicion']                   # atributos de arquero
tiene_gk = df[COLS_GK].notna().any(axis=1)                               # True si tiene algún atributo de arquero
sin_campo = df[COLS_CAMPO].isna().all(axis=1) & df['overall'].notna()    # True si no tiene ningún atributo de campo
es_gk_por_stats = df['linea'].isna() & (tiene_gk | sin_campo)            # candidatos a arquero sin posición cargada
df.loc[es_gk_por_stats, ['posicion', 'linea']] = ['GK', 'Arquero']       # los marca como arqueros
df['linea'] = df['linea'].fillna('Sin dato')                             # lo demás queda 'Sin dato' (ya NO se fuerza a Delantero)

# Protección: en algunos datasets el arquero trae ritmo/tiro/... con valores de estirada/manejo -> se anulan
df.loc[df['linea'] == 'Arquero', COLS_CAMPO] = np.nan                    # los arqueros no tienen atributos de campo

ORDEN_LINEAS = ['Arquero', 'Defensores', 'Mediocampistas', 'Delanteros', 'Sin dato']   # orden de presentación
df['linea'] = pd.Categorical(df['linea'], categories=ORDEN_LINEAS, ordered=True)       # categoría ordenada
print('✅ Tabla unificada lista:', f'{len(df):,}', 'filas')              # confirmación

In [ ]:
# --- 5. DIAGNÓSTICO: revisá estas salidas antes de publicar -------------------
print('1) Registros por edición y género')                               # título
print(df.groupby(['version', 'edicion', 'genero']).size().unstack(fill_value=0).to_string())   # cuántos jugadores hay en cada edición

print('\n2) % de datos faltantes por edición (hombres)')                 # título
hombres = df[df['genero'] == 'M']                                        # solo jugadores masculinos
faltantes = hombres.groupby('version').apply(                            # por edición...
    lambda g: pd.Series({'sin_liga': (g['liga'] == 'Desconocida').mean(),   # % sin liga
                         'sin_posicion': (g['linea'] == 'Sin dato').mean(), # % sin posición
                         'sin_nacionalidad': g['nacionalidad'].isna().mean(),   # % sin nacionalidad
                         'sin_club': g['club'].isna().mean()}))         # % sin club
print((faltantes * 100).round(1).to_string())                            # en porcentaje

print('\n3) Ligas más frecuentes de la última edición que NO se normalizaron (revisá si falta alguna regla)')   # título
ultima = hombres[hombres['version'] == hombres['version'].max()]         # última edición disponible
canonicas = {n for n, _ in REGLAS_LIGAS}                                 # nombres ya normalizados
print(ultima[~ultima['liga'].isin(canonicas)]['liga'].value_counts().head(20).to_string())   # las 20 más comunes sin regla

print('\n4) Liga Argentina: jugadores por edición')                      # título
conteo_arg = hombres[hombres['es_liga_arg']].groupby('version').size()      # jugadores argentinos por edición
for v in sorted(hombres['version'].unique()):                            # recorre todas las ediciones
    n = int(conteo_arg.get(v, 0))                                        # cantidad en esa edición (0 si no hay)
    print(f"   {etiqueta_edicion(v):8s} {n:5d} {'⚠️ SIN DATOS: revisá los nombres de liga de abajo' if n == 0 else ''}")   # marca las vacías
    if n == 0:                                                           # si falta, muestra los nombres de liga de esa edición
        print('      Ligas más frecuentes en esa edición:', hombres[hombres['version'] == v]['liga_original'].value_counts().head(15).to_dict())

print('\n5) Controles de calidad')                                       # título
arq = df[df['linea'] == 'Arquero']                                       # todos los arqueros
print('Arqueros con ritmo cargado (debe ser 0):', int(arq['ritmo'].notna().sum()))   # chequeo de la corrección
print('Filas con línea "Sin dato":', f"{(df['linea'] == 'Sin dato').sum():,}")        # cuántas quedaron sin clasificar

print('\n6) Mejor jugador de la Liga Argentina por línea (última edición)')   # título
arg_ult = ultima[ultima['es_liga_arg']].sort_values('overall', ascending=False)       # argentinos ordenados por overall
print(arg_ult.groupby('linea', observed=True).head(1)[['nombre', 'club', 'posicion', 'linea', 'overall']].to_string(index=False))   # el mejor de cada línea

In [ ]:
# --- 6. GUARDAR LA TABLA CONSOLIDADA ------------------------------------------
ruta_csv = f'{BASE}/tabla_general_jugadores_por_ano.csv'                 # ruta de salida
df.to_csv(ruta_csv, index=False)                                         # guarda el CSV sin la columna de índice
print('💾 Guardado:', ruta_csv)                                          # confirma

In [ ]:
# --- 7. PLANTILLA DEL DASHBOARD (HTML + CSS + JavaScript) ---------------------
# Es un texto plano con marcadores __PLAYERS__, __LIGAS__, __EDS__ y __KPI__ que la celda 8 reemplaza por los datos.
PLANTILLA_HTML = r"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Una década de fútbol en los videojuegos · FIFA 15 → EA FC 26</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  /* ===== Variables de color: cambiando estas líneas cambia todo el tema ===== */
  :root{--bg:#060a13;--panel:#0e1627;--panel2:#131d33;--line:#1f2b45;--txt:#b8c4d9;--white:#f4f7fb;
        --accent:#22d3ee;--accent2:#6366f1;--red:#ef4444;--gold:#facc15;}
  *{box-sizing:border-box}
  body{margin:0;background:var(--bg);color:var(--txt);font-family:Inter,'Segoe UI',system-ui,sans-serif;line-height:1.5}
  /* ===== Cabecera (hero) con degradado ===== */
  .hero{padding:56px 20px 36px;text-align:center;
        background:radial-gradient(1200px 400px at 50% -10%,rgba(99,102,241,.35),transparent),
                   radial-gradient(800px 300px at 90% 0%,rgba(34,211,238,.18),transparent),var(--bg);
        border-bottom:1px solid var(--line)}
  .eyebrow{letter-spacing:.2em;font-size:.75rem;color:var(--accent);margin:0 0 10px;font-weight:700}
  h1{margin:0;font-size:clamp(1.8rem,4.5vw,3.2rem);color:var(--white);font-weight:800;letter-spacing:-.02em}
  h1 span{background:linear-gradient(90deg,var(--accent),var(--accent2));-webkit-background-clip:text;background-clip:text;color:transparent}
  .lead{max-width:760px;margin:14px auto 0;font-size:1.05rem}
  /* ===== Tarjetas KPI ===== */
  .kpis{display:flex;flex-wrap:wrap;gap:14px;justify-content:center;margin-top:28px}
  .kpi{min-width:150px;padding:14px 20px;border-radius:14px;background:rgba(255,255,255,.04);border:1px solid var(--line)}
  .kpi b{display:block;font-size:1.6rem;color:var(--white)} .kpi small{color:var(--txt)}
  /* ===== Pestañas ===== */
  .tabs{position:sticky;top:0;z-index:20;display:flex;gap:8px;overflow-x:auto;padding:12px 20px;
        background:rgba(6,10,19,.85);backdrop-filter:blur(10px);border-bottom:1px solid var(--line)}
  .tab{flex:0 0 auto;padding:9px 16px;border-radius:999px;border:1px solid var(--line);background:var(--panel);
       color:var(--txt);cursor:pointer;font-weight:600;font-size:.92rem}
  .tab.on{background:linear-gradient(90deg,var(--accent2),var(--accent));color:#04101c;border-color:transparent}
  /* ===== Filtros ===== */
  .filters{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px;max-width:1200px;margin:22px auto 0;padding:0 20px}
  .filters label{font-size:.72rem;letter-spacing:.08em;text-transform:uppercase;color:var(--accent);font-weight:700}
  select,input{width:100%;margin-top:4px;padding:10px 12px;border-radius:10px;border:1px solid var(--line);background:var(--panel);color:var(--white);font-size:.95rem}
  /* ===== Contenedores ===== */
  main{max-width:1200px;margin:22px auto 60px;padding:0 20px}
  .view{display:none} .view.on{display:block}
  .card{background:linear-gradient(180deg,var(--panel2),var(--panel));border:1px solid var(--line);border-radius:16px;padding:22px;margin-bottom:22px;box-shadow:0 12px 40px rgba(0,0,0,.35)}
  .card h2{margin:0 0 4px;color:var(--white);font-size:1.25rem} .card p.sub{margin:0 0 14px;font-size:.92rem}
  .grid2{display:grid;grid-template-columns:1fr;gap:22px} @media(min-width:980px){.grid2{grid-template-columns:1fr 1fr}}
  .chart{width:100%;min-height:380px}
  .insight{margin:12px 0 0;padding:12px 14px;border-left:3px solid var(--accent);background:rgba(34,211,238,.07);border-radius:8px;color:var(--white)}
  /* ===== Tablas ===== */
  .tw{overflow-x:auto} table{width:100%;border-collapse:collapse;font-size:.92rem}
  th{padding:10px;text-align:left;font-size:.72rem;letter-spacing:.08em;text-transform:uppercase;color:var(--accent);border-bottom:1px solid var(--line)}
  td{padding:10px;border-bottom:1px solid var(--line)} tr:hover td{background:rgba(99,102,241,.08)}
  .badge{display:inline-block;min-width:34px;text-align:center;padding:3px 8px;border-radius:8px;font-weight:800;color:#04101c;background:var(--gold)}
  .pill{display:inline-block;padding:2px 9px;border-radius:999px;font-size:.75rem;font-weight:700;color:#04101c}
  /* ===== Selector de ligas (checkboxes) ===== */
  .checks{display:flex;flex-wrap:wrap;gap:8px;margin:6px 0 12px}
  .checks label{display:flex;gap:6px;align-items:center;padding:6px 12px;border:1px solid var(--line);border-radius:999px;background:var(--panel);cursor:pointer;font-size:.85rem;color:var(--txt)}
  .checks input{width:auto;margin:0}
  /* ===== Cancha para el equipo ideal ===== */
  .pitch{position:relative;width:100%;max-width:640px;aspect-ratio:3/4;margin:0 auto;border-radius:16px;border:2px solid rgba(255,255,255,.55);
         background:repeating-linear-gradient(0deg,#14532d 0 8.33%,#166534 8.33% 16.66%);overflow:hidden}
  .pitch:before{content:"";position:absolute;left:0;right:0;top:50%;border-top:2px solid rgba(255,255,255,.55)}
  .pitch:after{content:"";position:absolute;left:50%;top:50%;width:22%;aspect-ratio:1;transform:translate(-50%,-50%);border:2px solid rgba(255,255,255,.55);border-radius:50%}
  .chip{position:absolute;transform:translate(-50%,-50%);width:22%;text-align:center;z-index:2}
  .chip .ovr{display:inline-block;min-width:38px;padding:4px 8px;border-radius:10px;font-weight:900;color:#04101c;background:var(--gold);box-shadow:0 4px 12px rgba(0,0,0,.5)}
  .chip span{display:block;margin-top:3px;font-size:.78rem;font-weight:700;color:#fff;text-shadow:0 1px 3px #000;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
  .chip small{display:block;font-size:.66rem;color:#d1fae5;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
  footer{text-align:center;padding:36px 20px;border-top:1px solid var(--line);font-size:.9rem}
  footer a{color:var(--accent)}
</style>
</head>
<body>
<header class="hero">
  <p class="eyebrow">DATA STORYTELLING · FIFA 15 → EA FC 26</p>
  <h1>Una década de fútbol <span>en los videojuegos</span></h1>
  <p class="lead">Explorá cómo evolucionaron las ligas, los clubes y los jugadores a lo largo de 12 ediciones: filtrá, comparás jugadores y armá el equipo ideal de cualquier liga o club.</p>
  <div class="kpis" id="kpis"></div>
</header>

<nav class="tabs" id="tabs">
  <button class="tab on" data-tab="ligas">📈 Ligas</button>
  <button class="tab" data-tab="explorador">🔎 Explorador</button>
  <button class="tab" data-tab="xi">🧩 Equipo ideal</button>
  <button class="tab" data-tab="club">🏟️ Historia del club</button>
  <button class="tab" data-tab="comparador">⚔️ Comparador</button>
</nav>

<section class="filters" id="filters">
  <div><label for="fEd">Edición</label><select id="fEd"></select></div>
  <div><label for="fLiga">Liga</label><select id="fLiga"></select></div>
  <div><label for="fClub">Club</label><select id="fClub"></select></div>
  <div><label for="fLinea">Línea</label><select id="fLinea"></select></div>
</section>

<main>
  <!-- Vista 1: evolución de ligas (no usa los filtros de arriba) -->
  <div class="view on" id="v-ligas">
    <div class="card">
      <h2>Evolución del nivel de las ligas</h2>
      <p class="sub">Elegí las ligas y la métrica. "Top 100" compara a los 100 mejores de cada liga, lo que evita que pese el tamaño de la plantilla.</p>
      <div class="checks" id="ligaChecks"></div>
      <div style="max-width:320px"><label style="font-size:.72rem;letter-spacing:.08em;text-transform:uppercase;color:var(--accent);font-weight:700">Métrica</label>
        <select id="metrica"><option value="top100">Promedio de los 100 mejores</option><option value="media">Promedio de todos los jugadores</option></select></div>
      <div id="chartLigas" class="chart"></div>
      <p class="insight" id="insightLigas"></p>
    </div>
  </div>

  <!-- Vista 2: explorador de jugadores -->
  <div class="view" id="v-explorador">
    <div class="card">
      <h2>Ranking de jugadores</h2>
      <p class="sub" id="subExp"></p>
      <div style="max-width:280px;margin-bottom:12px"><label style="font-size:.72rem;letter-spacing:.08em;text-transform:uppercase;color:var(--accent);font-weight:700">Ordenar por</label><select id="statSel"></select></div>
      <div class="grid2">
        <div id="chartExp" class="chart"></div>
        <div class="tw"><table><thead><tr><th>#</th><th>Jugador</th><th>Club</th><th>Pos</th><th>OVR</th><th id="thStat"></th></tr></thead><tbody id="tbExp"></tbody></table></div>
      </div>
    </div>
  </div>

  <!-- Vista 3: equipo ideal 4-3-3 -->
  <div class="view" id="v-xi">
    <div class="card">
      <h2>Equipo ideal 4-3-3</h2>
      <p class="sub" id="subXI"></p>
      <div class="pitch" id="pitch"></div>
      <p class="insight" id="insightXI"></p>
    </div>
  </div>

  <!-- Vista 4: historia de un club -->
  <div class="view" id="v-club">
    <div class="card">
      <h2 id="tituloClub">Historia del club</h2>
      <p class="sub">Mejor jugador de cada edición para el club elegido en el filtro "Club".</p>
      <div id="chartClub" class="chart"></div>
    </div>
    <div class="card">
      <h2>Salón de la fama por línea</h2>
      <p class="sub">Máximo overall histórico del club en cada línea táctica.</p>
      <div class="tw"><table><thead><tr><th>Línea</th><th>Jugador</th><th>Edición</th><th>Overall</th></tr></thead><tbody id="tbClub"></tbody></table></div>
    </div>
  </div>

  <!-- Vista 5: comparador con radar -->
  <div class="view" id="v-comparador">
    <div class="card">
      <h2>Comparador de jugadores</h2>
      <p class="sub">Elegí dos jugadores de campo de la edición seleccionada (escribí para buscar).</p>
      <div class="grid2">
        <div><label style="font-size:.72rem;color:var(--accent);font-weight:700">JUGADOR A</label><input id="cmpA" list="dlCmp" autocomplete="off"></div>
        <div><label style="font-size:.72rem;color:var(--accent);font-weight:700">JUGADOR B</label><input id="cmpB" list="dlCmp" autocomplete="off"></div>
      </div>
      <datalist id="dlCmp"></datalist>
      <div class="grid2" style="margin-top:14px">
        <div id="chartRadar" class="chart"></div>
        <div class="tw"><table><thead><tr><th></th><th id="thA"></th><th id="thB"></th></tr></thead><tbody id="tbCmp"></tbody></table></div>
      </div>
    </div>
  </div>
</main>

<footer>
  Hecho con Python, Pandas y Plotly · Datasets públicos de valoraciones FIFA / EA FC (jugadores masculinos; incluye los 400 mejores de cada edición y toda la Liga Argentina) ·
  <a href="https://www.linkedin.com/in/tomas-monserrat-b14894125" target="_blank" rel="noopener">Tomás Monserrat</a>
</footer>

<script>
(function(){
  // ===== 1. Datos inyectados desde Python =====
  const RAW = __PLAYERS__;            // {cols:[...], rows:[[...],...]} jugadores
  const LIGAS = __LIGAS__;            // promedios por liga y edición
  const EDS = __EDS__;                // ediciones ordenadas de la más vieja a la más nueva
  const KPI = __KPI__;                // números para las tarjetas de la cabecera
  // Convierte cada fila (lista) en un objeto {nombre:..., overall:...} para trabajar cómodo
  const P = RAW.rows.map(r => { const o = {}; RAW.cols.forEach((c, i) => { o[c] = r[i]; }); return o; });

  // ===== 2. Constantes y utilidades =====
  const $ = id => document.getElementById(id);                                   // atajo para getElementById
  const esc = s => String(s).replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c])); // evita romper el HTML con comillas
  const LINEAS = ['Arquero','Defensores','Mediocampistas','Delanteros'];         // líneas tácticas
  const COL = {Arquero:'#facc15',Defensores:'#38bdf8',Mediocampistas:'#a78bfa',Delanteros:'#f87171','Sin dato':'#64748b'}; // color por línea
  const STATS = {overall:'Overall',potencial:'Potencial',ritmo:'Ritmo',tiro:'Tiro',pase:'Pase',regate:'Regate',defensa:'Defensa',fisico:'Físico'}; // atributos ordenables
  const RADAR = ['ritmo','tiro','pase','regate','defensa','fisico'];             // ejes del radar
  // Orden de prioridad de las ligas: las 5 grandes primero, después la Liga Profesional argentina, y luego el resto
  const PRIORIDAD = ['Premier League (Inglaterra)','LaLiga (España)','Serie A (Italia)','Bundesliga (Alemania)','Ligue 1 (Francia)','Liga Profesional (Argentina)'];
  const rangoLiga = l => { const i = PRIORIDAD.indexOf(l); return i === -1 ? 999 : i; };   // posición en la lista de prioridad (999 = sin prioridad)
  const S = {ed: EDS[EDS.length-1], liga:'Todas', club:'Todos', linea:'Todas', stat:'overall', tab:'ligas'}; // estado actual de la app
  const ax = extra => Object.assign({gridcolor:'#1f2b45', zerolinecolor:'#1f2b45'}, extra || {});         // estilo de ejes
  // Dibuja con Plotly usando un tema oscuro transparente
  function plot(id, data, layout){
    const base = {paper_bgcolor:'rgba(0,0,0,0)', plot_bgcolor:'rgba(0,0,0,0)', font:{color:'#b8c4d9'},
                  margin:{l:50,r:20,t:20,b:50}, legend:{orientation:'h', y:-0.2}, xaxis:ax(), yaxis:ax()};
    Plotly.react(id, data, Object.assign(base, layout || {}), {responsive:true, displayModeBar:false});
  }
  const uniq = a => [...new Set(a)].filter(x => x !== null && x !== undefined && x !== '');   // valores únicos sin vacíos
  const fmt = v => (v === null || v === undefined) ? '—' : v;                    // muestra — si no hay dato
  // Llena un <select> y conserva la selección previa si sigue existiendo
  function fill(id, values, current){
    const el = $(id);
    el.innerHTML = values.map(v => `<option value="${esc(v)}">${esc(v)}</option>`).join('');
    el.value = values.includes(current) ? current : values[0];
    return el.value;
  }
  // Jugadores que cumplen los filtros de arriba (edición, liga, club y línea)
  function filtrados(sinLinea){
    return P.filter(p => p.edicion === S.ed && (S.liga === 'Todas' || p.liga === S.liga) &&
      (S.club === 'Todos' || p.club === S.club) && (sinLinea || S.linea === 'Todas' || p.linea === S.linea));
  }

  // ===== 3. Filtros dependientes (liga -> club) =====
  function refrescarFiltros(){
    const enEd = P.filter(p => p.edicion === S.ed);                                // jugadores de la edición elegida
    const cuenta = {}; enEd.forEach(p => { if (p.liga) cuenta[p.liga] = (cuenta[p.liga] || 0) + 1; });
    const ligas = Object.keys(cuenta).sort((a, b) => rangoLiga(a) - rangoLiga(b) || cuenta[b] - cuenta[a]);   // 5 grandes, luego Argentina, luego el resto por cantidad de jugadores
    S.liga = fill('fLiga', ['Todas', ...ligas], S.liga);
    const clubes = uniq(enEd.filter(p => S.liga === 'Todas' || p.liga === S.liga).map(p => p.club)).sort();
    S.club = fill('fClub', ['Todos', ...clubes], S.club);
    S.linea = fill('fLinea', ['Todas', ...LINEAS], S.linea);
  }

  // ===== 4. Vista: evolución de ligas =====
  function vistaLigas(){
    const ligas = uniq(LIGAS.map(d => d.liga)).sort((a, b) => rangoLiga(a) - rangoLiga(b) || a.localeCompare(b));   // mismo orden de prioridad
    if (!$('ligaChecks').children.length){                                         // arma los checkboxes la primera vez
      const top5 = PRIORIDAD;                                                      // por defecto: las 5 grandes + la Liga Argentina
      $('ligaChecks').innerHTML = ligas.map(l => `<label><input type="checkbox" value="${esc(l)}" ${top5.includes(l) ? 'checked' : ''}> ${esc(l)}</label>`).join('');
      $('ligaChecks').onchange = vistaLigas; $('metrica').onchange = vistaLigas;
    }
    const sel = [...document.querySelectorAll('#ligaChecks input:checked')].map(i => i.value);
    const m = $('metrica').value;
    const trazas = sel.map(l => ({
      type:'scatter', mode:'lines+markers', name:l, connectgaps:false,
      x:EDS, y:EDS.map(e => { const d = LIGAS.find(z => z.liga === l && z.edicion === e); return d ? d[m] : null; })
    }));
    plot('chartLigas', trazas, {yaxis:ax({title:'Overall promedio'}), margin:{l:55,r:20,t:20,b:70}});
    const ult = EDS[EDS.length-1];                                                 // edición más reciente
    const rank = sel.map(l => { const d = LIGAS.find(z => z.liga === l && z.edicion === ult); return d ? {l, v:d[m]} : null; }).filter(Boolean).sort((a, b) => b.v - a.v);
    $('insightLigas').textContent = rank.length ? `En ${ult}, entre las ligas seleccionadas lidera ${rank[0].l} con ${rank[0].v.toFixed(1)}.` : 'Seleccioná al menos una liga.';
  }

  // ===== 5. Vista: explorador (tabla + barras) =====
  function vistaExplorador(){
    const stat = S.stat;
    const filas = filtrados().filter(p => p[stat] !== null && p[stat] !== undefined).sort((a, b) => b[stat] - a[stat]);
    $('subExp').textContent = `${filas.length.toLocaleString('es-AR')} jugadores en ${S.ed} con los filtros actuales. Los arqueros no tienen ritmo/tiro/pase, por eso solo aparecen al ordenar por Overall o Potencial.`;
    $('thStat').textContent = STATS[stat];
    $('tbExp').innerHTML = filas.slice(0, 25).map((p, i) => `<tr><td>${i + 1}</td><td><b style="color:#fff">${esc(p.nombre)}</b><br><small>${esc(fmt(p.nacionalidad))}</small></td><td>${esc(fmt(p.club))}</td><td><span class="pill" style="background:${COL[p.linea] || '#64748b'}">${esc(fmt(p.posicion))}</span></td><td><span class="badge">${fmt(p.overall)}</span></td><td>${fmt(p[stat])}</td></tr>`).join('');
    const top = filas.slice(0, 15).reverse();                                      // 15 mejores (invertidos para que el #1 quede arriba)
    plot('chartExp', [{type:'bar', orientation:'h', x:top.map(p => p[stat]), y:top.map(p => p.nombre),
      marker:{color:top.map(p => COL[p.linea] || '#64748b')}, text:top.map(p => p[stat]), textposition:'inside',
      hovertext:top.map(p => `${p.club || ''} · ${p.posicion || ''}`)}],
      {margin:{l:130,r:20,t:10,b:30}, xaxis:ax({range:[Math.max(0, (top.length ? top[0][stat] : 50) - 25), 100]})});
  }

  // ===== 6. Vista: equipo ideal 4-3-3 en una cancha =====
  const SLOTS = [   // cada puesto: línea, posiciones preferidas y coordenadas (x,y en %) sobre la cancha
    {l:'Arquero', pref:['GK'], x:50, y:90},
    {l:'Defensores', pref:['LB','LWB'], x:12, y:70}, {l:'Defensores', pref:['CB'], x:37, y:73},
    {l:'Defensores', pref:['CB'], x:63, y:73}, {l:'Defensores', pref:['RB','RWB'], x:88, y:70},
    {l:'Mediocampistas', pref:['CDM','CM'], x:50, y:54}, {l:'Mediocampistas', pref:['CM','CAM','LM'], x:27, y:42},
    {l:'Mediocampistas', pref:['CM','CAM','RM'], x:73, y:42},
    {l:'Delanteros', pref:['LW','LF'], x:15, y:19}, {l:'Delanteros', pref:['ST','CF','SS'], x:50, y:12}, {l:'Delanteros', pref:['RW','RF'], x:85, y:19}
  ];
  function vistaXI(){
    const pool = filtrados(true).sort((a, b) => b.overall - a.overall);            // candidatos ordenados por overall (ignora el filtro de línea)
    const usados = new Set(); const elegidos = [];
    SLOTS.forEach(s => {
      let p = pool.find(x => !usados.has(x) && x.linea === s.l && s.pref.includes(x.posicion));   // mejor jugador en la posición exacta
      if (!p) p = pool.find(x => !usados.has(x) && x.linea === s.l);                               // si no hay, mejor de la línea
      if (p) usados.add(p);
      elegidos.push({s, p});
    });
    const ambito = [S.liga !== 'Todas' ? S.liga : null, S.club !== 'Todos' ? S.club : null].filter(Boolean).join(' · ') || 'todo el dataset';
    $('subXI').textContent = `${S.ed} · ${ambito}`;
    $('pitch').innerHTML = elegidos.map(({s, p}) => p ?
      `<div class="chip" style="left:${s.x}%;top:${s.y}%"><div class="ovr">${p.overall}</div><span>${esc(p.nombre)}</span><small>${esc(fmt(p.club))}</small></div>` : '').join('');
    const con = elegidos.filter(e => e.p);
    $('insightXI').textContent = con.length ? `Media del equipo: ${(con.reduce((a, e) => a + e.p.overall, 0) / con.length).toFixed(1)} · ${con.length}/11 puestos cubiertos.` : 'No hay jugadores con estos filtros.';
  }

  // ===== 7. Vista: historia de un club =====
  function vistaClub(){
    const club = S.club;
    if (club === 'Todos'){                                                         // sin club elegido
      $('tituloClub').textContent = 'Elegí un club en el filtro "Club"';
      Plotly.purge('chartClub'); $('tbClub').innerHTML = ''; return;
    }
    $('tituloClub').textContent = `Historia de ${club}`;
    const mejores = EDS.map(e => {                                                 // mejor jugador del club en cada edición
      const js = P.filter(p => p.club === club && p.edicion === e).sort((a, b) => b.overall - a.overall);
      return js.length ? js[0] : null;
    });
    const pts = mejores.map((p, i) => p ? {e:EDS[i], p} : null).filter(Boolean);
    plot('chartClub', [{type:'scatter', mode:'lines+markers+text', x:pts.map(d => d.e), y:pts.map(d => d.p.overall),
      text:pts.map(d => d.p.nombre), textposition:'top center', line:{color:'#ef4444', width:3}, marker:{size:9}, textfont:{size:10}}],
      {yaxis:ax({title:'Overall máximo', range:[Math.min(...pts.map(d => d.p.overall)) - 4, Math.max(...pts.map(d => d.p.overall)) + 5]}), margin:{l:55,r:20,t:30,b:50}});
    $('tbClub').innerHTML = LINEAS.map(l => {                                      // mejor histórico por línea
      const js = P.filter(p => p.club === club && p.linea === l).sort((a, b) => b.overall - a.overall);
      return js.length ? `<tr><td><span class="pill" style="background:${COL[l]}">${l}</span></td><td><b style="color:#fff">${esc(js[0].nombre)}</b></td><td>${js[0].edicion}</td><td><span class="badge">${js[0].overall}</span></td></tr>` : '';
    }).join('');
  }

  // ===== 8. Vista: comparador con radar =====
  function vistaComparador(){
    const cand = P.filter(p => p.edicion === S.ed && p.ritmo !== null).sort((a, b) => b.overall - a.overall);   // solo jugadores de campo
    const etiqueta = p => `${p.nombre} · ${p.club || 's/club'}`;
    const mapa = {}; cand.forEach(p => { mapa[etiqueta(p)] = p; });
    $('dlCmp').innerHTML = cand.slice(0, 1500).map(p => `<option value="${esc(etiqueta(p))}">`).join('');
    const a = $('cmpA'), b = $('cmpB');
    if (!mapa[a.value]) a.value = cand[0] ? etiqueta(cand[0]) : '';                // valores por defecto: los dos mejores
    if (!mapa[b.value]) b.value = cand[1] ? etiqueta(cand[1]) : '';
    const A = mapa[a.value], B = mapa[b.value];
    if (!A || !B){ return; }
    const cerrar = arr => arr.concat([arr[0]]);                                    // cierra el polígono del radar
    const ejes = cerrar(RADAR.map(k => STATS[k]));
    plot('chartRadar', [
      {type:'scatterpolar', r:cerrar(RADAR.map(k => A[k])), theta:ejes, fill:'toself', name:A.nombre, line:{color:'#22d3ee'}},
      {type:'scatterpolar', r:cerrar(RADAR.map(k => B[k])), theta:ejes, fill:'toself', name:B.nombre, line:{color:'#f87171'}}
    ], {polar:{bgcolor:'rgba(0,0,0,0)', radialaxis:{range:[0, 100], gridcolor:'#1f2b45'}, angularaxis:{gridcolor:'#1f2b45'}}, margin:{l:40,r:40,t:20,b:40}});
    $('thA').textContent = A.nombre; $('thB').textContent = B.nombre;
    const filas = [['Club', A.club, B.club], ['Posición', A.posicion, B.posicion], ['Overall', A.overall, B.overall], ['Potencial', A.potencial, B.potencial]]
      .concat(RADAR.map(k => [STATS[k], A[k], B[k]]));
    $('tbCmp').innerHTML = filas.map(f => `<tr><td>${f[0]}</td><td>${esc(fmt(f[1]))}</td><td>${esc(fmt(f[2]))}</td></tr>`).join('');
  }

  // ===== 9. Navegación y render general =====
  const VISTAS = {ligas:vistaLigas, explorador:vistaExplorador, xi:vistaXI, club:vistaClub, comparador:vistaComparador};
  function render(){
    document.querySelectorAll('.view').forEach(v => v.classList.toggle('on', v.id === 'v-' + S.tab));   // muestra solo la vista activa
    document.querySelectorAll('.tab').forEach(t => t.classList.toggle('on', t.dataset.tab === S.tab));  // resalta la pestaña
    $('filters').style.display = S.tab === 'ligas' ? 'none' : 'grid';              // la vista de ligas no usa filtros
    VISTAS[S.tab]();                                                               // dibuja la vista (ya visible, así Plotly mide bien)
  }

  // ===== 10. Eventos =====
  document.querySelectorAll('.tab').forEach(t => { t.onclick = () => { S.tab = t.dataset.tab; render(); }; });
  $('fEd').onchange = e => { S.ed = e.target.value; refrescarFiltros(); render(); };
  $('fLiga').onchange = e => { S.liga = e.target.value; S.club = 'Todos'; refrescarFiltros(); render(); };
  $('fClub').onchange = e => { S.club = e.target.value; render(); };
  $('fLinea').onchange = e => { S.linea = e.target.value; render(); };
  $('statSel').onchange = e => { S.stat = e.target.value; render(); };
  $('cmpA').onchange = render; $('cmpB').onchange = render;

  // ===== 11. Arranque =====
  $('kpis').innerHTML = KPI.map(k => `<div class="kpi"><b>${esc(k.v)}</b><small>${esc(k.l)}</small></div>`).join('');
  $('statSel').innerHTML = Object.keys(STATS).map(k => `<option value="${k}">${STATS[k]}</option>`).join('');
  $('fEd').innerHTML = EDS.slice().reverse().map(e => `<option value="${e}">${e}</option>`).join(''); $('fEd').value = S.ed;
  // Valores iniciales: Liga Argentina y River Plate (si existen en la última edición)
  const ligasUlt = uniq(P.filter(p => p.edicion === S.ed).map(p => p.liga));
  if (ligasUlt.includes('Liga Profesional (Argentina)')) S.liga = 'Liga Profesional (Argentina)';
  const river = uniq(P.filter(p => p.edicion === S.ed && (S.liga === 'Todas' || p.liga === S.liga)).map(p => p.club)).find(c => /river plate/i.test(c));
  if (river) S.club = river;
  refrescarFiltros();
  render();
})();
</script>
</body>
</html>"""

In [ ]:
# --- 8. PREPARAR LOS DATOS Y GENERAR EL HTML ----------------------------------
masc = df[df['genero'] == 'M'].copy()                                     # el dashboard usa jugadores masculinos

# 8.1 Jugadores que viajan dentro del HTML: top 400 de cada edición + TODA la Liga Argentina
top_global = masc.sort_values('overall', ascending=False).groupby('edicion').head(400)   # 400 mejores por edición
web = pd.concat([top_global, masc[masc['es_liga_arg']]]).drop_duplicates(['player_id', 'edicion'])   # + Liga Argentina, sin repetir
web = web[web['overall'].notna()].copy()                                  # sin overall no sirve para el dashboard
web['linea'] = web['linea'].astype(str)                                   # categoría -> texto para JSON
cols_web = ['nombre', 'edicion', 'liga', 'club', 'posicion', 'linea', 'overall', 'potencial',
            'ritmo', 'tiro', 'pase', 'regate', 'defensa', 'fisico', 'nacionalidad']   # columnas que usa el HTML
web = web[cols_web]                                                       # nos quedamos solo con esas
for c in ['overall', 'potencial', 'ritmo', 'tiro', 'pase', 'regate', 'defensa', 'fisico']:   # atributos numéricos
    web[c] = web[c].round().astype('Int64')                               # enteros que admiten vacío
datos = json.loads(web.to_json(orient='split', force_ascii=False))        # pandas -> estructura JSON (NaN pasa a null)
players_json = json.dumps({'cols': datos['columns'], 'rows': datos['data']}, ensure_ascii=False, separators=(',', ':'))   # JSON compacto

# 8.2 Promedios por liga y edición (calculados sobre TODA la tabla, no solo sobre lo que viaja en el HTML)
LIGAS_DASH = [n for n, _ in REGLAS_LIGAS]                                 # las 10 ligas normalizadas
agg = (masc[masc['liga'].isin(LIGAS_DASH) & masc['overall'].notna()]
       .groupby(['liga', 'edicion', 'version'])['overall']
       .agg(media='mean', n='count', top100=lambda s: s.nlargest(100).mean())   # promedio general y de los 100 mejores
       .reset_index().sort_values(['liga', 'version']))                   # ordenado por liga y edición
agg[['media', 'top100']] = agg[['media', 'top100']].round(2)              # dos decimales
ligas_json = json.dumps(agg[['liga', 'edicion', 'media', 'top100', 'n']].to_dict('records'), ensure_ascii=False)   # lista de objetos

# 8.3 Ediciones ordenadas y tarjetas KPI de la cabecera
eds = masc[['edicion', 'version']].drop_duplicates().sort_values('version')['edicion'].tolist()   # de FIFA 15 a FC 26
mejor = masc[masc['version'] == masc['version'].max()].sort_values('overall', ascending=False).iloc[0]   # mejor jugador de la última edición
kpi = [{'v': f'{len(masc):,}'.replace(',', '.'), 'l': 'registros de jugadores'},           # total de registros
       {'v': str(len(eds)), 'l': 'ediciones analizadas'},                                   # cantidad de ediciones
       {'v': str(masc.loc[masc['liga'] != 'Desconocida', 'liga'].nunique()), 'l': 'ligas distintas'},   # cantidad de ligas
       {'v': f"{mejor['nombre']} ({int(mejor['overall'])})", 'l': f"mejor jugador de {mejor['edicion']}"}]   # el mejor de hoy

# 8.4 Reemplazar los marcadores de la plantilla y guardar el archivo
def seguro(texto):
    """Evita que un '</script>' dentro de los datos cierre el script del HTML."""
    return texto.replace('</', '<\\/')                                    # escapa la secuencia peligrosa

html = (PLANTILLA_HTML
        .replace('__PLAYERS__', seguro(players_json))                     # jugadores
        .replace('__LIGAS__', seguro(ligas_json))                         # ligas
        .replace('__EDS__', json.dumps(eds))                              # ediciones
        .replace('__KPI__', seguro(json.dumps(kpi, ensure_ascii=False)))) # KPIs
ruta_html = f'{BASE}/dashboard_interactivo_futbol.html'                   # archivo de salida
with open(ruta_html, 'w', encoding='utf-8') as f:                         # lo escribe en UTF-8
    f.write(html)                                                         # contenido final
print(f'✅ HTML generado: {ruta_html} ({len(html) / 1e6:.2f} MB, {len(web):,} jugadores embebidos)')   # resumen

# 8.5 Descargar (solo funciona en Colab)
try:
    from google.colab import files                                        # módulo exclusivo de Colab
    files.download(ruta_html)                                             # descarga el HTML
except ImportError:
    print('Fuera de Colab: el archivo quedó en', ruta_html)               # en local, solo avisa la ruta